In [ ]:

import pandas as pd

# 定义训练集和测试集的路径
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

# 读取训练数据和测试数据
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 查看训练数据的前几行
train_df.head()


Displayed on the user's machine.

In [ ]:

# 检查训练数据和测试数据的基本信息
print("训练数据集信息:")
train_df.info()

print("\n测试数据集信息:")
test_df.info()

# 检查训练数据的缺失值
print("\n训练数据缺失值情况:")
train_df.isnull().sum()

# 检查测试数据的缺失值
print("\n测试数据缺失值情况:")
test_df.isnull().sum()


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

64  
 14  no_of_previous_cancellations          33674 non-null  int64  
 15  no_of_previous_bookings_not_canceled  33674 non-null  int64  
 16  avg_price_per_room                    33674 non-null  float64
 17  no_of_special_requests                33674 non-null  int64  
 18  booking_status                        33674 non-null  int64  
dtypes: float64(1), int64(18)
memory usage: 4.9 MB

测试数据集信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8420 entries, 0 to 8419
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    8420 non-null   int64  
 1   no_of_adults                          8420 non-null   int64  
 2   no_of_children                        8420 non-null  

In [ ]:


# 分离特征和目标变量
X_train = train_df.drop(columns=['booking_status', 'id'])
y_train = train_df['booking_status']

X_test = test_df.drop(columns=['booking_status', 'id'])
y_test = test_df['booking_status']

# 检查分离后的数据
X_train.shape, y_train.shape, X_test.shape, y_test.shape



((33674, 17), (33674,), (8420, 17), (8420,))

In [ ]:


# 检查特征列的数据类型
X_train.dtypes

# 检查是否有需要编码的类别变量
categorical_features = [col for col in X_train.columns if X_train[col].dtype == 'object']
categorical_features



[]

In [ ]:


import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 定义LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)

# 定义LightGBM参数
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# 训练模型
model = lgb.train(params, train_data, num_boost_round=100)

# 预测测试集
y_pred_prob = model.predict(X_test)

# 计算AUC-ROC
auc_roc = roc_auc_score(y_test, y_pred_prob)
auc_roc




np.float64(0.8935701434042931)

In [ ]:


# 创建预测结果的DataFrame
predictions_df = pd.DataFrame({
    'id': test_df['id'],
    'predicted_probability': y_pred_prob
})

# 保存预测结果到文件
output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/predictions.csv'
predictions_df.to_csv(output_path, index=False)

# 检查保存的文件
predictions_df.head()


Displayed on the user's machine.